In [ ]:
from pathlib import Path
RESULTS_CANDIDATES = [Path('../../results'), Path('../results'), Path('results'), Path('evaluation/results')]
RESULTS = next((p for p in RESULTS_CANDIDATES if p.exists()), Path('../../results'))
print(f'RESULTS = {RESULTS.resolve()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# NSDI style – Wong colorblind-safe palette
COLORS = {'Baseline': '#0072B2', 'Confidential5G-TDX': '#009E73'}
LIGHT_COLORS = {'Baseline': '#B3D9F2', 'Confidential5G-TDX': '#B3E5D1'}
DEPS   = ['Baseline', 'Confidential5G-TDX']
COL_W  = 3.33  # \columnwidth in inches for 10pt twocolumn
TEXT_W = 7.16  # \textwidth in inches
FIG_H  = 2.4
IMAGES = Path('images')
IMAGES.mkdir(exist_ok=True)

# 1:1 font mapping - 8pt here = 8pt in paper
FONT = {'tick': 10, 'label': 10, 'title': 10, 'legend': 8}

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.6,
    'grid.linewidth': 0.4,
    'hatch.linewidth': 0.5,
})

def iqr(s):   return s.quantile(0.75) - s.quantile(0.25)
def stats(s): return s.median(), iqr(s), s.quantile(0.95), s.std()

In [ ]:
# Input File Configuration
# Uncomment and set RESULTS to override the auto-detected path:
# RESULTS = Path('/absolute/path/to/results')

STARTUP_FILES = {
    'Baseline':           'nf_startup_baseline.csv',
    'Confidential5G-TDX': 'nf_startup_hybrid.csv',
}
REG_FILES = {
    'Baseline':           'cp_registration_baseline_2.csv',
    'Confidential5G-TDX': 'cp_registration_hybrid_3.csv',
}
SR_FILES = {
    'Baseline':           'cp_service_request_baseline.csv',
    'Confidential5G-TDX': 'cp_service_request_hybrid.csv',
}
NF_FILES = {
    'Baseline':           'cp_registration_baseline_2_nf.csv',
    'Confidential5G-TDX': 'cp_registration_hybrid_3_nf.csv',
}
IFACE_FILES = {
    'Baseline':         'cp_registration_baseline_2_iface.csv',
    'Confidential5G-TDX': 'cp_registration_hybrid_3_iface.csv',
}

In [ ]:
import matplotlib.ticker as ticker

# NF Startup Latency
startup = {}
for name, fname in STARTUP_FILES.items():
    df = pd.read_csv(RESULTS / fname)
    startup[name] = df[df['success'] == True].groupby('nf')[['nf_init_ms', 'nrf_reg_ms']].mean()

nfs = ['nrf'] + sorted(nf for nf in set().union(*[set(df.index) for df in startup.values()]) if nf != 'nrf')

x  = np.arange(len(nfs)) * 1.4  # wider spacing between NF groups
bw = 0.4

fig, ax = plt.subplots(figsize=(COL_W, FIG_H))
for i, dep in enumerate(DEPS):
    xi = x + (i - 0.5) * bw
    y1 = np.array([startup[dep].loc[nf, 'nf_init_ms'] if nf in startup[dep].index else 0 for nf in nfs])
    y2 = np.array([startup[dep].loc[nf, 'nrf_reg_ms'] if nf in startup[dep].index else 0 for nf in nfs])
    ax.bar(xi, y1, bw, color=COLORS[dep], edgecolor='white', linewidth=0.4, zorder=3)
    ax.bar(xi, y2, bw, bottom=y1, color=LIGHT_COLORS[dep], edgecolor='white', linewidth=0.4, hatch='//', zorder=3)

ax.tick_params(axis='both', labelsize=FONT['tick'])
ax.set_yscale('log'); ax.set_ylim(10, 10**4)
ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=15))
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)
ax.yaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=15))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())
ax.set_xticks(x)
ax.set_xticklabels([nf.upper() for nf in nfs], fontsize=FONT['tick'], rotation=35, ha='right')
ax.set_ylabel('Startup Time (ms)', fontsize=FONT['label'])

legend_handles = []
for dep in DEPS:
    short = 'Baseline' if dep == 'Baseline' else 'C5G-TDX'
    legend_handles.append(mpatches.Patch(facecolor=COLORS[dep], edgecolor='white', label=f'{short} init'))
    legend_handles.append(mpatches.Patch(facecolor=LIGHT_COLORS[dep], edgecolor='white', hatch='//', label=f'{short} NRF reg'))
ax.legend(handles=legend_handles, ncol=2, fontsize=FONT['legend'], frameon=False,
          loc='lower center', bbox_to_anchor=(0.5, 1.01))
ax.grid(axis='y', linestyle='-', alpha=0.15, zorder=0); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(IMAGES / 'cp_startup_latency.png', bbox_inches='tight', dpi=300)
fig.savefig(IMAGES / 'cp_startup_latency.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# NF Startup Latency – summary table (mean)
startup_mean = {}
for name, fname in STARTUP_FILES.items():
    df = pd.read_csv(RESULTS / fname)
    startup_mean[name] = df[df['success'] == True].groupby('nf')[['nf_init_ms', 'nrf_reg_ms']].mean()

rows = []
for nf in nfs:
    row = {'NF': nf.upper()}
    for dep in DEPS:
        df = startup_mean[dep]
        init = df.loc[nf, 'nf_init_ms'] if nf in df.index else float('nan')
        reg  = df.loc[nf, 'nrf_reg_ms'] if nf in df.index else float('nan')
        row[f'{dep} init (ms)'] = round(init, 2)
        row[f'{dep} nrf_reg (ms)'] = round(reg, 2)
        row[f'{dep} total (ms)'] = round(init + reg, 2)
    rows.append(row)

startup_table = pd.DataFrame(rows).set_index('NF')
display(startup_table)

In [ ]:
# Registration Latency – load
reg_data = {}
for name, fname in REG_FILES.items():
    df = pd.read_csv(RESULTS / fname)
    reg_data[name] = df[df['success'] == True].reset_index(drop=True)
    print(f'{name}: {len(reg_data[name])} rows')

In [ ]:
import matplotlib.patches as mpatches

order = ['Baseline', 'C5G-TDX']
_key = {'Baseline': 'Baseline', 'C5G-TDX': 'Confidential5G-TDX'}
other_color = '#B8C0CC'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(COL_W, 2.8), gridspec_kw={'wspace': 0.65})

bar_width = 0.45
x = np.arange(len(order))
reg_meds = [reg_data[_key[n]]['reg_time_ms'].median() for n in order]
pdu_meds = [reg_data[_key[n]]['pdu_time_ms'].median() for n in order]
total_series = [pd.to_numeric(reg_data[_key[n]]['total_attach_ms'], errors='coerce').dropna() for n in order]
total_meds = [s.median() for s in total_series]
total_q1 = [s.quantile(0.25) for s in total_series]
total_q3 = [s.quantile(0.75) for s in total_series]
residual = [max(0.0, t - r - p) for t, r, p in zip(total_meds, reg_meds, pdu_meds)]
total_yerr = [
    [m - q1 for m, q1 in zip(total_meds, total_q1)],
    [q3 - m for m, q3 in zip(total_meds, total_q3)],
]

for i, n in enumerate(order):
    k = _key[n]
    ax1.bar(x[i], reg_meds[i], bar_width, color=COLORS[k], edgecolor='white', linewidth=0.6, zorder=3)
    ax1.bar(x[i], pdu_meds[i], bar_width, bottom=reg_meds[i], color=LIGHT_COLORS[k], edgecolor='white', linewidth=0.6, zorder=3)
    ax1.bar(x[i], pdu_meds[i], bar_width, bottom=reg_meds[i], color='none', edgecolor='#777', linewidth=0.0, hatch='//', zorder=3)
    ax1.bar(x[i], residual[i], bar_width, bottom=reg_meds[i] + pdu_meds[i],
            color=other_color, edgecolor='none', linewidth=0.0, zorder=3)

ax1.errorbar(x, total_meds, yerr=total_yerr, fmt='none', ecolor='#555', elinewidth=1.0, capsize=3, zorder=4)
ax1.set_xticks(x)
ax1.set_xticklabels(order, fontsize=FONT['tick'], rotation=25, ha='right')
ax1.set_ylabel('Latency (ms)', fontsize=FONT['label'])
ax1.set_ylim(0, 600)
ax1.tick_params(labelsize=FONT['tick'])
ax1.set_title('(a)', fontsize=FONT['title'], pad=4)
ax1.grid(axis='y', linestyle='-', alpha=0.2, zorder=0); ax1.set_axisbelow(True)

if 'sr_data' not in globals() or not isinstance(sr_data, dict):
    sr_data = {}
for name, fname in SR_FILES.items():
    if name in sr_data: continue
    p = RESULTS / fname
    if not p.exists(): continue
    df = pd.read_csv(p)
    sr_data[name] = df[df['success'] == True].reset_index(drop=True)

deps = [d for d in order if _key[d] in sr_data and not sr_data[_key[d]].empty]
sr_series = [pd.to_numeric(sr_data[_key[d]]['service_request_ms'], errors='coerce').dropna() for d in deps]
sr_med = [s.median() for s in sr_series]
sr_q1 = [s.quantile(0.25) for s in sr_series]
sr_q3 = [s.quantile(0.75) for s in sr_series]
sr_yerr = [[m - q1 for m, q1 in zip(sr_med, sr_q1)], [q3 - m for m, q3 in zip(sr_med, sr_q3)]]

x2 = np.arange(len(deps))
ax2.bar(x2, sr_med, 0.45, color=[COLORS[_key[d]] for d in deps], edgecolor='white', linewidth=0.6, zorder=3)
ax2.errorbar(x2, sr_med, yerr=sr_yerr, fmt='none', ecolor='#555', elinewidth=1.0, capsize=3, zorder=4)
ax2.set_xticks(x2)
ax2.set_xticklabels(deps, fontsize=FONT['tick'], rotation=25, ha='right')
ax2.set_ylabel('Latency (ms)', fontsize=FONT['label'])
ax2.set_ylim(0, 90)
ax2.tick_params(labelsize=FONT['tick'])
ax2.set_title('(b)', fontsize=FONT['title'], pad=4)
ax2.grid(axis='y', linestyle='-', alpha=0.2, zorder=0); ax2.set_axisbelow(True)

# Shared legend above both subplots
legend_handles = [
    mpatches.Patch(facecolor=COLORS['Baseline'], edgecolor='white', label='Base Reg'),
    mpatches.Patch(facecolor=LIGHT_COLORS['Baseline'], edgecolor='#777', hatch='//', label='Base PDU'),
    mpatches.Patch(facecolor=COLORS['Confidential5G-TDX'], edgecolor='white', label='C5G Reg'),
    mpatches.Patch(facecolor=LIGHT_COLORS['Confidential5G-TDX'], edgecolor='#777', hatch='//', label='C5G PDU'),
]
fig.legend(handles=legend_handles, fontsize=FONT['legend']-1, ncol=4, loc='upper center',
           bbox_to_anchor=(0.5, 1.08), frameon=False, columnspacing=0.4, handlelength=1.0, handletextpad=0.3)

fig.tight_layout()
fig.savefig(IMAGES / 'cp_reg_and_sr.png', bbox_inches='tight', dpi=300)
fig.savefig(IMAGES / 'cp_reg_and_sr.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Registration – stats table
metrics = [('reg_time_ms', 'Registration'), ('pdu_time_ms', 'PDU Session'), ('total_attach_ms', 'Total Attach')]
print(f"{'Deployment':<22} {'Metric':<16} {'Median':>8} {'Std':>8} {'IQR':>8} {'p95':>8}  (ms)")
print('─' * 75)
for name, df in reg_data.items():
    for col, label in metrics:
        med, iq, p95, std = stats(df[col])
        print(f'{name:<22} {label:<16} {med:8.2f} {std:8.2f} {iq:8.2f} {p95:8.2f}')
    print()

In [ ]:
# Service Request Latency – load
sr_data = {}
for name, fname in SR_FILES.items():
    p = RESULTS / fname
    if not p.exists(): print(f'Missing: {fname}'); continue
    df = pd.read_csv(p)
    sr_data[name] = df[df['success'] == True].reset_index(drop=True)
    print(f'{name}: {len(sr_data[name])} rows')

In [ ]:
# Service Request – distribution stats
if not sr_data:
    print('No service request data available.')
else:
    print(f"{'Deployment':<22} {'Mean':>8} {'Median':>8} {'Std':>8} {'IQR':>8} {'Skew':>8} {'p95':>8}  (ms)")
    print('─' * 80)
    for name, df in sr_data.items():
        s = df['service_request_ms']
        med, iq, p95, std = stats(s)
        print(f'{name:<22} {s.mean():8.2f} {med:8.2f} {std:8.2f} {iq:8.2f} {s.skew():8.2f} {p95:8.2f}')

In [ ]:
# NF Processing – load
dfs_nf = {}
for name, fname in NF_FILES.items():
    if fname is None: continue
    p = RESULTS / fname
    if not p.exists(): print(f'Missing: {fname}'); continue
    dfs_nf[name] = pd.read_csv(p)
    print(f'{name} cols: {[c for c in dfs_nf[name].columns if c.endswith("_ms")]}')

In [ ]:
# NF Processing – 2-panel plot (AMF decomposition + other NFs)
def _parse_nf_cols(df):
    out = {}
    for c in df.columns:
        if not c.endswith('_ms'): continue
        for m in ['processing', 'waiting', 'forward', 'total']:
            if c.endswith(f'_{m}_ms'):
                out.setdefault(c[:-len(f'_{m}_ms')], {})[m] = c
                break
    return out

def _nf_total(df, nf, nf_map):
    specs = nf_map.get(nf, {})
    if 'total' in specs:
        return pd.to_numeric(df[specs['total']], errors='coerce')
    parts = [pd.to_numeric(df[specs[m]], errors='coerce') for m in ['processing', 'waiting', 'forward'] if m in specs]
    return pd.concat(parts, axis=1).sum(axis=1) if parts else pd.Series(np.nan, index=df.index)

def med_q(s, dep):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if s.empty: return (np.nan, np.nan, np.nan)
    return (s.median(), s.quantile(0.25), s.quantile(0.75))

if not dfs_nf:
    print('No NF data available.')
else:
    deps = [d for d in DEPS if d in dfs_nf]
    nf_maps = {d: _parse_nf_cols(dfs_nf[d]) for d in deps}

    amf_sbi = {
        d: med_q(pd.to_numeric(dfs_nf[d][nf_maps[d]['amf']['waiting']], errors='coerce'), d)
        if 'waiting' in nf_maps[d].get('amf', {}) else (0.0, 0.0, 0.0)
        for d in deps
    }
    amf_nonsbi = {
        d: med_q(pd.to_numeric(dfs_nf[d][nf_maps[d]['amf']['processing']], errors='coerce'), d)
        if 'processing' in nf_maps[d].get('amf', {}) else med_q(_nf_total(dfs_nf[d], 'amf', nf_maps[d]), d)
        for d in deps
    }
    amf_total = {d: med_q(_nf_total(dfs_nf[d], 'amf', nf_maps[d]), d) for d in deps}

    all_nfs = sorted(set().union(*[set(nf_maps[d].keys()) for d in deps]))
    other_nfs = [nf for nf in all_nfs if nf not in {'amf', 'scp', 'udr', 'upf'}]
    nf_stats = {nf: {d: med_q(_nf_total(dfs_nf[d], nf, nf_maps[d]), d) for d in deps} for nf in other_nfs}

    fig = plt.figure(figsize=(COL_W, 2.8))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 2], wspace=0.65)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    # Panel 1: AMF decomposition
    x1 = np.arange(len(deps))
    bw1 = 0.45
    ns_vals = [amf_nonsbi[d][0] if pd.notna(amf_nonsbi[d][0]) else 0.0 for d in deps]
    sb_vals = [amf_sbi[d][0] if pd.notna(amf_sbi[d][0]) else 0.0 for d in deps]
    tot_vals = [amf_total[d][0] for d in deps]
    tot_q1 = [amf_total[d][1] for d in deps]
    tot_q3 = [amf_total[d][2] for d in deps]
    tot_yerr = [
        [(m - q1) if pd.notna(m) and pd.notna(q1) else np.nan for m, q1 in zip(tot_vals, tot_q1)],
        [(q3 - m) if pd.notna(m) and pd.notna(q3) else np.nan for m, q3 in zip(tot_vals, tot_q3)],
    ]

    for i, dep in enumerate(deps):
        ax1.bar(x1[i], ns_vals[i], bw1, color=COLORS[dep], edgecolor='white', linewidth=0.6, zorder=3)
        ax1.bar(x1[i], sb_vals[i], bw1, bottom=ns_vals[i], color=LIGHT_COLORS[dep], edgecolor='white', linewidth=0.6, zorder=3)
        ax1.bar(x1[i], sb_vals[i], bw1, bottom=ns_vals[i], color='none', edgecolor='#777', linewidth=0.0, hatch='//', zorder=3)

    ax1.errorbar(x1, tot_vals, yerr=tot_yerr, fmt='none', ecolor='#555', elinewidth=1.0, capsize=3, zorder=4)
    ax1.set_xticks(x1)
    ax1.set_xticklabels(['Baseline' if d == 'Baseline' else 'C5G-TDX' for d in deps], fontsize=FONT['tick'], rotation=25, ha='right')
    ax1.set_ylabel('Latency (ms)', fontsize=FONT['label'])
    ax1.set_title('(a)', fontsize=FONT['title'], pad=4)
    max_amf = max((v + (q3 - v if pd.notna(v) and pd.notna(q3) else 0.0) for v, q3 in zip(tot_vals, tot_q3) if pd.notna(v)), default=1)
    ax1.set_ylim(0, max(600, max_amf * 1.22 if np.isfinite(max_amf) else 1))
    ax1.tick_params(labelsize=FONT['tick'])

    legend_handles = []
    for dep in deps:
        short = 'Baseline' if dep == 'Baseline' else 'C5G-TDX'
        legend_handles.append(mpatches.Patch(facecolor=COLORS[dep], edgecolor='white', label=f'{short} proc'))
        legend_handles.append(mpatches.Patch(facecolor=LIGHT_COLORS[dep], edgecolor='#666', hatch='//', label=f'{short} wait'))
    fig.legend(handles=legend_handles, fontsize=FONT['legend']-1, ncol=4, loc='upper center',
               bbox_to_anchor=(0.5, 1.08), frameon=False, columnspacing=0.4, handlelength=1.0, handletextpad=0.3)
    ax1.grid(axis='y', linestyle='-', alpha=0.2, zorder=0); ax1.set_axisbelow(True)

    # Panel 2: other NFs
    x2 = np.arange(len(other_nfs))
    bw2 = 0.4
    for i, dep in enumerate(deps):
        meds = [nf_stats[nf][dep][0] for nf in other_nfs]
        q1s = [nf_stats[nf][dep][1] for nf in other_nfs]
        q3s = [nf_stats[nf][dep][2] for nf in other_nfs]
        yerr = [
            [(m - q1) if pd.notna(m) and pd.notna(q1) else np.nan for m, q1 in zip(meds, q1s)],
            [(q3 - m) if pd.notna(m) and pd.notna(q3) else np.nan for m, q3 in zip(meds, q3s)],
        ]
        dep_x = x2 + (i - 0.5) * bw2
        vals = [m if pd.notna(m) else 0.0 for m in meds]
        ax2.bar(dep_x, vals, bw2, color=COLORS[dep], edgecolor='white', linewidth=0.6, zorder=3)
        ax2.errorbar(dep_x, meds, yerr=yerr, fmt='none', ecolor='#555', elinewidth=1.0, capsize=3, zorder=4)

    ax2.set_xticks(x2)
    ax2.set_xticklabels([nf.upper() for nf in other_nfs], fontsize=FONT['tick'], rotation=35, ha='right')
    ax2.set_ylabel('Latency (ms)', fontsize=FONT['label'])
    ax2.set_title('(b)', fontsize=FONT['title'], pad=4)
    ax2.set_ylim(0, 30)
    ax2.tick_params(labelsize=FONT['tick'])

    legend_handles_2 = [mpatches.Patch(facecolor=COLORS[d], edgecolor='white',
                        label='Baseline' if d == 'Baseline' else 'C5G-TDX') for d in deps]
    ax2.legend(handles=legend_handles_2, fontsize=FONT['legend'], frameon=False)
    ax2.grid(axis='y', linestyle='-', alpha=0.2, zorder=0); ax2.set_axisbelow(True)

    fig.tight_layout()
    fig.savefig(IMAGES / 'cp_nf_processing.png', bbox_inches='tight', dpi=300)
    fig.savefig(IMAGES / 'cp_nf_processing.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# NF processing decomposition summary table (median/IQR)
if '_parse_nf_cols' not in globals():
    def _parse_nf_cols(df):
        out = {}
        for c in df.columns:
            if not c.endswith('_ms'):
                continue
            for m in ['processing', 'waiting', 'forward', 'total']:
                if c.endswith(f'_{m}_ms'):
                    out.setdefault(c[:-len(f'_{m}_ms')], {})[m] = c
                    break
        return out

if '_nf_total' not in globals():
    def _nf_total(df, nf, nf_map):
        specs = nf_map.get(nf, {})
        if 'total' in specs:
            return pd.to_numeric(df[specs['total']], errors='coerce')
        parts = [pd.to_numeric(df[specs[m]], errors='coerce') for m in ['processing', 'waiting', 'forward'] if m in specs]
        return pd.concat(parts, axis=1).sum(axis=1) if parts else pd.Series(np.nan, index=df.index)

def med_iqr(s):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if s.empty:
        return (0.0, 0.0)
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    return (s.median(), q3 - q1)

if not dfs_nf:
    print('No NF data available.')
else:
    deps = [d for d in DEPS if d in dfs_nf]
    nf_maps = {d: _parse_nf_cols(dfs_nf[d]) for d in deps}

    all_nfs = sorted(set().union(*[set(nf_maps[d].keys()) for d in deps]))
    focus_nfs = [nf for nf in all_nfs if nf not in {'scp', 'nrf', 'udr'}]

    rows = []
    for nf in focus_nfs:
        for dep in deps:
            df = dfs_nf[dep]
            specs = nf_maps[dep].get(nf, {})

            proc = pd.to_numeric(df[specs['processing']], errors='coerce').dropna() if 'processing' in specs else pd.Series(dtype=float)
            wait = pd.to_numeric(df[specs['waiting']], errors='coerce').dropna() if 'waiting' in specs else pd.Series(dtype=float)
            total = _nf_total(df, nf, nf_maps[dep]).dropna()

            if proc.empty and wait.empty and total.empty:
                continue

            proc_med, proc_iqr = med_iqr(proc)
            wait_med, wait_iqr = med_iqr(wait)
            total_med, total_iqr = med_iqr(total)

            rows.append({
                'NF': nf.upper(),
                'Deployment': dep,
                'Processing median (ms)': proc_med,
                'Waiting median (ms)': wait_med,
                'Total median (ms)': total_med,
                'Processing IQR (ms)': proc_iqr,
                'Waiting IQR (ms)': wait_iqr,
                'Total IQR (ms)': total_iqr,
                'Waiting share (%)': (wait_med / total_med * 100.0) if total_med > 0 else np.nan,
                'Samples': int(len(total)) if not total.empty else 0,
            })

    summary = pd.DataFrame(rows).sort_values(['NF', 'Deployment']).reset_index(drop=True)

    rounded = summary.copy()
    for col in ['Processing median (ms)', 'Waiting median (ms)', 'Total median (ms)',
                'Processing IQR (ms)', 'Waiting IQR (ms)', 'Total IQR (ms)', 'Waiting share (%)']:
        rounded[col] = rounded[col].round(2)

    try:
        display(rounded)
    except NameError:
        print(rounded.to_string(index=False))



In [ ]:
# Interface Latency
IFACE_LABELS = {
    'n2_auth_ms':     'N2 Auth',    'n2_auth_rtt_ms':  'N2 Auth',
    'n2_security_ms': 'N2 SecMode', 'n2_secmode_rtt_ms': 'N2 SecMode',
    'n4_session_ms':  'N4 Session',
    'sbi_ausf_ms':    'AUSF SBI',   'sbi_udm_ms': 'UDM SBI',
    'sbi_smf_ms':     'SMF SBI',    'sbi_pcf_ms': 'PCF SBI',
    'n2_ping_ms':     'N2 Ping',    'n2_path_ping_ms': 'N2 Ping',
    'n3_ping_ms':     'N3 Ping',    'n3_path_ping_ms': 'N3 Ping',
}

records = []
for deployment, fname in IFACE_FILES.items():
    if fname is None: continue
    p = RESULTS / fname
    if not p.exists(): print(f'Missing: {fname}'); continue
    df = pd.read_csv(p)
    seen = set()
    for col, label in IFACE_LABELS.items():
        if col not in df.columns or label in seen: continue
        seen.add(label)
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        med, iq, p95, _ = stats(s)
        records.append({'Deployment': deployment, 'Interface': label,
                        'Median (ms)': round(med,2), 'IQR (ms)': round(iq,2), 'p95 (ms)': round(p95,2)})

df_iface = pd.DataFrame(records).set_index(['Deployment','Interface'])
try:
    display(df_iface)
except NameError:
    print(df_iface.to_string())